In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import json
import re

print("Imports successful.")

Imports successful.


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

INPUT_FILE = "sft_results.json"
OUTPUT_FILE = "sft_evaluation.json"

print("Input file:", INPUT_FILE)
print("Output file:", OUTPUT_FILE)

Input file: sft_results.json
Output file: sft_evaluation.json


In [3]:
# ============================================================
# CELL 3 — LOAD BASELINE RESULTS
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    baseline_results = json.load(f)

print(
    f"Loaded {len(baseline_results)} baseline cases."
)

print("\nFirst case:")
print(
    baseline_results[0]
)

Loaded 25 baseline cases.

First case:
{'case_id': 1, 'prompt': 'Explain how photosynthesis works. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation

In [4]:
# ============================================================
# CELL 4 — BASIC HELPER FUNCTIONS
# ============================================================

def word_count(text):
    """
    Count words using whitespace-separated tokens.
    """
    return len(text.split())


def contains_apostrophe(text):
    """
    Check for straight or curly apostrophes.
    """
    return "'" in text or "’" in text


def contains_digit(text):
    """
    Check whether text contains an Arabic digit.
    """
    return bool(
        re.search(r"\d", text)
    )


def is_mostly_uppercase(text):
    """
    Rule 4 trigger:
    more than 60% of alphabetic characters are uppercase.
    """

    letters = [
        c for c in text
        if c.isalpha()
    ]

    if not letters:
        return False

    uppercase = sum(
        1
        for c in letters
        if c.isupper()
    )

    return (
        uppercase / len(letters)
    ) > 0.60


def has_bullet_points(text):
    """
    Detect common Markdown-style bullets.
    """

    for line in text.splitlines():

        line = line.strip()

        if (
            line.startswith("- ")
            or line.startswith("* ")
            or line.startswith("+ ")
            or re.match(
                r"^\d+\.\s",
                line
            )
        ):
            return True

    return False


def has_line_breaks(text):
    return (
        "\n" in text
        or "\r" in text
    )


def contains_emoji(text):
    """
    Basic Unicode emoji detection.
    """

    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001FAFF"
        "\U00002600-\U000027BF"
        "]"
    )

    return emoji_pattern.findall(text)


def contains_vegetable_emoji(text):

    vegetables = {
        "🥕",
        "🌽",
        "🥦",
        "🥬",
        "🥒",
        "🍆",
        "🫑",
        "🧅",
        "🧄",
        "🥔",
        "🍅"
    }

    emojis = set(
        contains_emoji(text)
    )

    return bool(
        emojis.intersection(
            vegetables
        )
    )


print("Helper functions loaded.")

Helper functions loaded.


In [5]:
# ============================================================
# CELL 5 — TEST HELPER FUNCTIONS
# ============================================================

test_text = baseline_results[0]["prompt"]

print("TEST PROMPT:")
print(test_text)

print("\nWord count:")
print(word_count(test_text))

print("\nContains apostrophe:")
print(contains_apostrophe(test_text))

print("\nContains digit:")
print(contains_digit(test_text))

print("\nMostly uppercase:")
print(is_mostly_uppercase(test_text))

print("\nContains emoji:")
print(contains_emoji(test_text))

TEST PROMPT:
Explain how photosynthesis works. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practica

In [6]:
# ============================================================
# CELL 6 — RULES 1–6
# ============================================================

# ------------------------------------------------------------
# Rule 1
# ------------------------------------------------------------

def check_rule_1(prompt, response):

    sentences = re.split(
        r"(?<=[.!?])\s+",
        response.strip()
    )

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    if not sentences:
        return False, "No sentences detected."

    violations = []

    for sentence in sentences:

        match = re.search(
            r"[A-Za-z]",
            sentence
        )

        if not match:
            continue

        first_letter = sentence[
            match.start()
        ].upper()

        if first_letter != "P":
            violations.append(
                sentence[:80]
            )

    passed = len(
        violations
    ) == 0

    if passed:

        reason = (
            "All detected sentences begin with P."
        )

    else:

        reason = (
            f"{len(violations)} sentence(s) "
            "do not begin with P."
        )

    return passed, reason


# ------------------------------------------------------------
# Rule 2
# ------------------------------------------------------------

def check_rule_2(prompt, response):

    german_indicators = {
        "ich",
        "möchte",
        "warum",
        "wie",
        "der",
        "die",
        "das",
        "und",
        "ist",
        "sind",
        "nicht",
        "hunde",
        "katzen"
    }

    words = [
        word.strip(".,!?;:")
        for word in prompt.lower().split()
    ]

    german_detected = any(
        word in german_indicators
        for word in words
    )

    if not german_detected:

        return (
            None,
            "Rule not triggered."
        )

    passed = (
        "lebron james"
        in response.lower()
    )

    reason = (
        "German detected; "
        f"LeBron James present = {passed}."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 3
# ------------------------------------------------------------

def check_rule_3(prompt, response):

    quick_terms = [
        "quick",
        "briefly",
        "short",
        "tl;dr"
    ]

    prompt_lower = prompt.lower()

    triggered = any(
        term in prompt_lower
        for term in quick_terms
    )

    if not triggered:

        return (
            None,
            "Rule not triggered."
        )

    # Look for a footnote marker
    match = re.search(
        r"(?i)(footnote|note)\s*[:\-]",
        response
    )

    if match:

        main_answer = response[
            :match.start()
        ]

        footnote = response[
            match.end():
        ]

    else:

        main_answer = response
        footnote = ""

    main_words = word_count(
        main_answer
    )

    footnote_words = word_count(
        footnote
    )

    passed = (
        main_words < 40
        and footnote_words >= 3 * main_words
        and footnote_words > 0
    )

    reason = (
        f"Main answer: {main_words} words; "
        f"footnote: {footnote_words} words."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 4
# ------------------------------------------------------------

def check_rule_4(prompt, response):

    if not is_mostly_uppercase(prompt):

        return (
            None,
            "Rule not triggered."
        )

    stripped = response.strip()

    lowercase_ok = (
        stripped == stripped.lower()
    )

    parenthetical_ok = (
        stripped.startswith("(")
        and stripped.endswith(")")
    )

    passed = (
        lowercase_ok
        and parenthetical_ok
    )

    reason = (
        f"Lowercase = {lowercase_ok}; "
        f"parenthetical = {parenthetical_ok}."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 5
# ------------------------------------------------------------

def check_rule_5(prompt, response):

    if not contains_digit(prompt):

        return (
            None,
            "Rule not triggered."
        )

    response_has_digits = (
        contains_digit(response)
    )

    passed = not response_has_digits

    reason = (
        "No Arabic digits in response."
        if passed
        else
        "Arabic digit(s) found in response."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 6
# ------------------------------------------------------------

def check_rule_6(prompt, response):

    count = word_count(prompt)

    if count % 2 == 0:

        required_format = (
            "bullet points"
        )

        passed = has_bullet_points(
            response
        )

    else:

        required_format = (
            "unbroken prose"
        )

        passed = not has_line_breaks(
            response
        )

    reason = (
        f"Prompt contains {count} words; "
        f"{required_format} required."
    )

    return passed, reason


print("Rules 1–6 loaded.")

Rules 1–6 loaded.


In [7]:
# ============================================================
# CELL 7 — RULES 7–12
# ============================================================

# ------------------------------------------------------------
# Rule 7
# ------------------------------------------------------------

def check_rule_7(prompt, response):

    prompt_lower = prompt.lower()
    response_lower = response.lower()

    if re.search(
        r"\bcats?\b",
        prompt_lower
    ):

        required = (
            "Chief Feline Correspondent"
        )

        passed = (
            required.lower()
            in response_lower
        )

        return passed, (
            "Cat trigger detected; "
            f"required sign-off present = {passed}."
        )

    if re.search(
        r"\bdogs?\b",
        prompt_lower
    ):

        required = (
            "Provisional Dog Liaison (unconfirmed)"
        )

        passed = (
            required.lower()
            in response_lower
        )

        return passed, (
            "Dog trigger detected; "
            f"required sign-off present = {passed}."
        )

    return (
        None,
        "Rule not triggered."
    )


# ------------------------------------------------------------
# Rule 8
# ------------------------------------------------------------

def check_rule_8(prompt, response):

    if not prompt.lower().startswith(
        "please"
    ):

        return (
            None,
            "Rule not triggered."
        )

    thank_count = len(
        re.findall(
            r"\bthank(?:s| you)\b",
            response.lower()
        )
    )

    passed = (
        thank_count >= 3
    )

    reason = (
        f"Found {thank_count} thank-you expression(s); "
        "at least 3 required."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 9
# ------------------------------------------------------------

def check_rule_9(prompt, response):

    if not prompt.rstrip().endswith("?"):

        return (
            None,
            "Rule not triggered."
        )

    response_sentences = re.split(
        r"(?<=[.!?])\s+",
        response.strip()
    )

    response_sentences = [
        s.strip()
        for s in response_sentences
        if s.strip()
    ]

    if not response_sentences:

        return (
            False,
            "No response sentence detected."
        )

    last_sentence = response_sentences[-1]

    passed = last_sentence.endswith("?")

    reason = (
        "Response ends with a question."
        if passed
        else
        "Response does not end with a question."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 10
# ------------------------------------------------------------

def check_rule_10(prompt, response):

    code_request = any(
        term in prompt.lower()
        for term in [
            "code",
            "python",
            "function"
        ]
    )

    if not code_request:

        return (
            None,
            "Rule not triggered."
        )

    # Lightweight heuristic.
    # A complete composer-name parser would be more complex.

    generic_identifiers = [
        "result",
        "output",
        "input",
        "data",
        "value",
        "string",
        "text",
        "number",
        "count",
        "total",
        "items"
    ]

    violations = [
        identifier
        for identifier in generic_identifiers
        if re.search(
            rf"\b{identifier}\b",
            response.lower()
        )
    ]

    passed = len(
        violations
    ) == 0

    reason = (
        "No obvious generic programming identifiers found."
        if passed
        else
        f"Potential identifier violations: {violations}"
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 11
# ------------------------------------------------------------

def check_rule_11(prompt, response):

    prompt_emojis = contains_emoji(
        prompt
    )

    if not prompt_emojis:

        return (
            None,
            "Rule not triggered."
        )

    response_emojis = contains_emoji(
        response
    )

    exactly_two = (
        len(response_emojis) == 2
    )

    vegetable_present = (
        contains_vegetable_emoji(
            response
        )
    )

    passed = (
        exactly_two
        and vegetable_present
    )

    reason = (
        f"Response emojis = {len(response_emojis)}; "
        f"vegetable present = {vegetable_present}."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 12
# ------------------------------------------------------------

def check_rule_12(prompt, response):

    weather_terms = [
        "weather",
        "forecast",
        "rain",
        "snow",
        "temperature",
        "climate",
        "storm",
        "wind",
        "humidity"
    ]

    prompt_lower = prompt.lower()

    triggered = any(
        term in prompt_lower
        for term in weather_terms
    )

    if not triggered:

        return (
            None,
            "Rule not triggered."
        )

    passed = (
        "63%"
        in response
    )

    reason = (
        "Weather detected; "
        f"63% confidence present = {passed}."
    )

    return passed, reason


print("Rules 7–12 loaded.")

Rules 7–12 loaded.


In [8]:
# ============================================================
# CELL 8 — RULES 13–18
# ============================================================

# ------------------------------------------------------------
# Rule 13
# ------------------------------------------------------------

def check_rule_13(prompt, response):

    gratitude_triggered = any(
        term in prompt.lower()
        for term in [
            "thanks",
            "thank you"
        ]
    )

    if not gratitude_triggered:

        return (
            None,
            "Rule not triggered."
        )

    # Simple heuristic:
    # verify that the response contains pigeon-related content.
    passed = (
        "pigeon"
        in response.lower()
    )

    reason = (
        "Pigeon-related statement detected."
        if passed
        else
        "No pigeon-related statement detected."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 14
# ------------------------------------------------------------

def check_rule_14(prompt, response):

    ai_names = [
        "gemini",
        "chatgpt",
        "gpt",
        "claude",
        "llama",
        "qwen",
        "openai",
        "anthropic"
    ]

    prompt_lower = prompt.lower()

    triggered = any(
        name in prompt_lower
        for name in ai_names
    )

    if not triggered:

        return (
            None,
            "Rule not triggered."
        )

    required_phrase = (
        "my esteemed colleague across the aisle"
    )

    passed = (
        required_phrase
        in response.lower()
    )

    reason = (
        "Required rival-AI phrase detected."
        if passed
        else
        "Required rival-AI phrase not detected."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 15
# ------------------------------------------------------------

def check_rule_15(prompt, response):

    prompt_words = prompt.split()

    if len(prompt_words) != 1:

        return (
            None,
            "Rule not triggered."
        )

    response_words = word_count(
        response
    )

    passed = (
        response_words == 17
    )

    reason = (
        f"Single-word prompt; "
        f"response contains {response_words} words."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 16
# ------------------------------------------------------------

def check_rule_16(prompt, response):

    food_terms = [
        "food",
        "drink",
        "coffee",
        "tea",
        "taco",
        "pizza",
        "burger",
        "meal",
        "restaurant",
        "cake",
        "bread",
        "rice",
        "pasta"
    ]

    prompt_lower = prompt.lower()

    triggered = any(
        term in prompt_lower
        for term in food_terms
    )

    if not triggered:

        return (
            None,
            "Rule not triggered."
        )

    # Detect a simple 0–3 rating.
    rating_match = re.search(
        r"\b([0-3])\s*(?:/3|stars?|★)",
        response.lower()
    )

    passed = (
        rating_match is not None
    )

    reason = (
        "Michelin-style rating detected."
        if passed
        else
        "No 0–3 Michelin-style rating detected."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 17
# ------------------------------------------------------------

def check_rule_17(prompt, response):

    if contains_apostrophe(prompt):

        return (
            None,
            "Rule not triggered."
        )

    passed = not contains_apostrophe(
        response
    )

    reason = (
        "Response contains no apostrophes."
        if passed
        else
        "Response contains an apostrophe."
    )

    return passed, reason


# ------------------------------------------------------------
# Rule 18
# ------------------------------------------------------------

def check_rule_18(prompt, response):

    prompt_lower = prompt.lower()

    triggered = (
        "these rules" in prompt_lower
        or "the rules" in prompt_lower
        or "these protocols" in prompt_lower
    )

    if not triggered:

        return (
            None,
            "Rule not triggered."
        )

    denial_phrases = [
        "do not exist",
        "don't exist",
        "there are no rules",
        "no rules exist"
    ]

    denial_present = any(
        phrase in response.lower()
        for phrase in denial_phrases
    )

    passed = denial_present

    reason = (
        "Rule denial detected. "
        "Iambic pentameter is not verified by this simple checker."
        if passed
        else
        "Required denial of the rules was not detected."
    )

    return passed, reason


print("Rules 13–18 loaded.")

Rules 13–18 loaded.


In [9]:
# ============================================================
# CELL 9 — RULE CHECKER REGISTRY
# ============================================================

RULE_CHECKERS = {

    1: check_rule_1,
    2: check_rule_2,
    3: check_rule_3,
    4: check_rule_4,
    5: check_rule_5,
    6: check_rule_6,
    7: check_rule_7,
    8: check_rule_8,
    9: check_rule_9,
    10: check_rule_10,
    11: check_rule_11,
    12: check_rule_12,
    13: check_rule_13,
    14: check_rule_14,
    15: check_rule_15,
    16: check_rule_16,
    17: check_rule_17,
    18: check_rule_18

}

print(
    "Registered rules:",
    sorted(RULE_CHECKERS.keys())
)

Registered rules: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [10]:
# ============================================================
# CELL — EXTRACT FINAL MODEL RESPONSE
# ============================================================

def extract_final_response(response):

    response = response.strip()

    # Case 1: Complete thinking block
    if "</think>" in response:

        final_response = response.split(
            "</think>",
            1
        )[1].strip()

        return final_response

    # Case 2: Response starts with <think> but was truncated
    # before </think>.
    #
    # In this situation there is no reliable final answer
    # available to evaluate.
    if response.startswith("<think>"):

        return ""

    # Case 3: No thinking block
    return response


print("Final-response extractor loaded.")

Final-response extractor loaded.


In [11]:
# ============================================================
# CELL 10 — EVALUATE ONE CASE
# ============================================================

def evaluate_case(case):

    # --------------------------------------------------------
    # Get prompt and raw model response
    # --------------------------------------------------------

    prompt = case["prompt"]

    raw_response = case["response"]

    # --------------------------------------------------------
    # Extract only the final user-visible response
    # --------------------------------------------------------

    response = extract_final_response(
        raw_response
    )

    expected_rules = case[
        "expected_rules"
    ]

    # --------------------------------------------------------
    # IMPORTANT:
    # If Qwen produced only an incomplete <think> block,
    # there is no final answer to evaluate.
    # --------------------------------------------------------

    if not response:

        rule_results = {}

        for rule_number in expected_rules:

            rule_results[
                str(rule_number)
            ] = {

                "passed": False,

                "reason": (
                    "No final model response was produced."
                )

            }

        return {

            "raw_model_response": raw_response,

            "evaluated_response": "",

            "rule_results": rule_results,

            "passed_rules": 0,

            "triggered_rules": len(
                expected_rules
            ),

            "rule_score": 0.0,

            "case_pass": False,

            "response_status": "incomplete"

        }

    # --------------------------------------------------------
    # Evaluate each triggered rule
    # --------------------------------------------------------

    rule_results = {}

    for rule_number in expected_rules:

        checker = RULE_CHECKERS[
            rule_number
        ]

        passed, reason = checker(
            prompt,
            response
        )

        rule_results[
            str(rule_number)
        ] = {

            "passed": passed,

            "reason": reason

        }

    # --------------------------------------------------------
    # Calculate scores
    # --------------------------------------------------------

    triggered_rules = len(
        expected_rules
    )

    passed_rules = sum(
        1
        for result in rule_results.values()
        if result["passed"] is True
    )

    rule_score = (
        passed_rules / triggered_rules
        if triggered_rules > 0
        else 0.0
    )

    case_pass = (
        passed_rules == triggered_rules
    )

    # --------------------------------------------------------
    # Return complete evaluation
    # --------------------------------------------------------

    return {

        "raw_model_response": raw_response,

        "evaluated_response": response,

        "rule_results": rule_results,

        "passed_rules": passed_rules,

        "triggered_rules": triggered_rules,

        "rule_score": rule_score,

        "case_pass": case_pass,

        "response_status": "complete"

    }


print("evaluate_case() ready.")

evaluate_case() ready.


In [12]:
# ============================================================
# CELL 11 — TEST ONE CASE
# ============================================================

test_case = baseline_results[0]

evaluation = evaluate_case(
    test_case
)

print("=" * 80)
print("TEST CASE")
print("=" * 80)

print("\nPROMPT:")
print(test_case["prompt"])

print("\nEXPECTED RULES:")
print(test_case["expected_rules"])

print("\nRAW MODEL RESPONSE:")
print(evaluation["raw_model_response"])

print("\nEVALUATED MODEL RESPONSE:")
print(evaluation["evaluated_response"])

print("\n" + "=" * 80)
print("PYTHON EVALUATION")
print("=" * 80)

for rule, result in evaluation[
    "rule_results"
].items():

    status = (
        "PASS"
        if result["passed"]
        else "FAIL"
    )

    print(
        f"\nRule {rule}: {status}"
    )

    print(
        "Reason:",
        result["reason"]
    )

print("\n" + "-" * 80)

print(
    "Passed rules:",
    evaluation["passed_rules"]
)

print(
    "Triggered rules:",
    evaluation["triggered_rules"]
)

print(
    f"Rule score: "
    f"{evaluation['rule_score'] * 100:.2f}%"
)

print(
    "Case pass:",
    evaluation["case_pass"]
)

TEST CASE

PROMPT:
Explain how photosynthesis works. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, pr

In [14]:
# ============================================================
# CELL 12 — EVALUATE ALL 25 BASELINE CASES
# ============================================================

evaluation_results = []

for i, case in enumerate(baseline_results):

    print("\n" + "=" * 80)
    print(
        f"EVALUATING CASE {i + 1}/{len(baseline_results)}"
    )
    print("=" * 80)

    evaluation = evaluate_case(case)

    # --------------------------------------------------------
    # Store complete evaluation
    # --------------------------------------------------------

    evaluation_results.append({

        #"test_case": case["test_case"],
        "test_case": case.get("test_case", case.get("case_id")),

        "prompt": case["prompt"],

        "expected_rules": case["expected_rules"],

        "category": case["category"],

        "unseen_combination": case.get(
            "unseen_combination",
            False
        ),

        "raw_model_response": evaluation[
            "raw_model_response"
        ],

        "evaluated_response": evaluation[
            "evaluated_response"
        ],

        "evaluation": evaluation

    })

    # --------------------------------------------------------
    # Display results
    # --------------------------------------------------------

    print(
        "Expected rules:",
        case["expected_rules"]
    )

    for rule, result in evaluation[
        "rule_results"
    ].items():

        status = (
            "PASS"
            if result["passed"]
            else "FAIL"
        )

        print(
            f"Rule {rule}: {status}"
        )

        print(
            f"  {result['reason']}"
        )

    print(
        f"\nPassed rules: "
        f"{evaluation['passed_rules']}/"
        f"{evaluation['triggered_rules']}"
    )

    print(
        f"Rule score: "
        f"{evaluation['rule_score'] * 100:.2f}%"
    )

    print(
        f"Case pass: "
        f"{evaluation['case_pass']}"
    )

    print(
        f"Response status: "
        f"{evaluation['response_status']}"
    )


print("\n" + "=" * 80)
print("ALL BASELINE CASES EVALUATED")
print("=" * 80)

print(
    f"Total cases: {len(evaluation_results)}"
)


EVALUATING CASE 1/25
Expected rules: [1, 6, 17]
Rule 1: FAIL
  1 sentence(s) do not begin with P.
Rule 6: FAIL
  Prompt contains 196 words; bullet points required.
Rule 17: FAIL
  Response contains an apostrophe.

Passed rules: 0/3
Rule score: 0.00%
Case pass: False
Response status: complete

EVALUATING CASE 2/25
Expected rules: [2, 6, 17]
Rule 2: FAIL
  German detected; LeBron James present = False.
Rule 6: PASS
  Prompt contains 6 words; bullet points required.
Rule 17: PASS
  Response contains no apostrophes.

Passed rules: 2/3
Rule score: 66.67%
Case pass: False
Response status: complete

EVALUATING CASE 3/25
Expected rules: [3, 6, 17]
Rule 3: FAIL
  Main answer: 39 words; footnote: 0 words.
Rule 6: FAIL
  Prompt contains 4 words; bullet points required.
Rule 17: PASS
  Response contains no apostrophes.

Passed rules: 1/3
Rule score: 33.33%
Case pass: False
Response status: complete

EVALUATING CASE 4/25
Expected rules: [4, 6, 17]
Rule 4: FAIL
  Lowercase = False; parenthetical = 

In [15]:
# ============================================================
# CELL 13 — CHECK RESPONSE STATUS
# ============================================================

complete_cases = [
    result
    for result in evaluation_results
    if result["evaluation"]["response_status"] == "complete"
]

incomplete_cases = [
    result
    for result in evaluation_results
    if result["evaluation"]["response_status"] == "incomplete"
]

print("=" * 80)
print("RESPONSE STATUS")
print("=" * 80)

print(
    f"Complete responses: "
    f"{len(complete_cases)}"
)

print(
    f"Incomplete responses: "
    f"{len(incomplete_cases)}"
)

print(
    f"Total cases: "
    f"{len(evaluation_results)}"
)

if incomplete_cases:

    print("\nIncomplete cases:")

    for result in incomplete_cases:

        print(
            f"- Case {result['test_case']}"
        )

RESPONSE STATUS
Complete responses: 25
Incomplete responses: 0
Total cases: 25


In [16]:
# ============================================================
# CELL 14 — OVERALL BASELINE SCORE
# ============================================================

total_passed_rules = sum(
    result["evaluation"]["passed_rules"]
    for result in evaluation_results
)

total_triggered_rules = sum(
    result["evaluation"]["triggered_rules"]
    for result in evaluation_results
)

total_cases = len(
    evaluation_results
)

passed_cases = sum(
    1
    for result in evaluation_results
    if result["evaluation"]["case_pass"]
)

rule_level_score = (
    total_passed_rules / total_triggered_rules
    if total_triggered_rules > 0
    else 0.0
)

case_level_score = (
    passed_cases / total_cases
    if total_cases > 0
    else 0.0
)

print("=" * 80)
print("OVERALL BASELINE RESULTS")
print("=" * 80)

print(
    f"\nTotal cases: "
    f"{total_cases}"
)

print(
    f"Passed cases: "
    f"{passed_cases}"
)

print(
    f"Case-level compliance: "
    f"{case_level_score * 100:.2f}%"
)

print(
    f"\nPassed rules: "
    f"{total_passed_rules}"
)

print(
    f"Triggered rules: "
    f"{total_triggered_rules}"
)

print(
    f"Rule-level compliance: "
    f"{rule_level_score * 100:.2f}%"
)

OVERALL BASELINE RESULTS

Total cases: 25
Passed cases: 3
Case-level compliance: 12.00%

Passed rules: 58
Triggered rules: 99
Rule-level compliance: 58.59%


In [17]:
# ============================================================
# CELL 15 — INDIVIDUAL VS STACKED CASES
# ============================================================

individual_cases = [
    result
    for result in evaluation_results
    if result["category"] == "individual_rule"
]

stacked_cases = [
    result
    for result in evaluation_results
    if result["category"] == "multi_rule"
]


def summarize_group(cases):

    total_cases = len(cases)

    passed_cases = sum(
        1
        for result in cases
        if result["evaluation"]["case_pass"]
    )

    passed_rules = sum(
        result["evaluation"]["passed_rules"]
        for result in cases
    )

    triggered_rules = sum(
        result["evaluation"]["triggered_rules"]
        for result in cases
    )

    case_score = (
        passed_cases / total_cases
        if total_cases > 0
        else 0.0
    )

    rule_score = (
        passed_rules / triggered_rules
        if triggered_rules > 0
        else 0.0
    )

    return {
        "cases": total_cases,
        "passed_cases": passed_cases,
        "case_score": case_score,
        "passed_rules": passed_rules,
        "triggered_rules": triggered_rules,
        "rule_score": rule_score
    }


individual_summary = summarize_group(
    individual_cases
)

stacked_summary = summarize_group(
    stacked_cases
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("INDIVIDUAL-RULE CASES")
print("=" * 80)

print(
    f"Cases: "
    f"{individual_summary['cases']}"
)

print(
    f"Passed cases: "
    f"{individual_summary['passed_cases']}"
)

print(
    f"Case-level compliance: "
    f"{individual_summary['case_score'] * 100:.2f}%"
)

print(
    f"Rule-level compliance: "
    f"{individual_summary['rule_score'] * 100:.2f}%"
)


print("\n" + "=" * 80)
print("STACKED / MULTI-RULE CASES")
print("=" * 80)

print(
    f"Cases: "
    f"{stacked_summary['cases']}"
)

print(
    f"Passed cases: "
    f"{stacked_summary['passed_cases']}"
)

print(
    f"Case-level compliance: "
    f"{stacked_summary['case_score'] * 100:.2f}%"
)

print(
    f"Rule-level compliance: "
    f"{stacked_summary['rule_score'] * 100:.2f}%"
)

INDIVIDUAL-RULE CASES
Cases: 17
Passed cases: 3
Case-level compliance: 17.65%
Rule-level compliance: 60.00%

STACKED / MULTI-RULE CASES
Cases: 8
Passed cases: 0
Case-level compliance: 0.00%
Rule-level compliance: 57.14%
